# Gemma Graph Extractor

Minimal notebook to load a bounded document and return a Pydantic-validated
graph. Supported input types in this version: `pdf`, `txt`, `md`.


In [28]:
from __future__ import annotations

import json
import os
import re
import uuid
from pathlib import Path

import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "config.py").exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from pydantic import BaseModel, Field
from pypdf import PdfReader

from core.config import APP_NAME, USER_ID


In [29]:
GOOGLE_GEMINI = (
    os.getenv("GOOGLE_GEMINI_API_KEY")
    or os.getenv("GOOGLE_GEMINI")
    or os.getenv("GEMINI_API_KEY")
)

if not GOOGLE_GEMINI:
    raise RuntimeError("Missing GOOGLE_GEMINI / GOOGLE_GEMINI_API_KEY in environment.")

MODEL_NAME = "gemini/gemma-4-31b-it"
model = LiteLlm(model=MODEL_NAME, api_key=GOOGLE_GEMINI)


In [30]:
class Evidence(BaseModel):
    page: int | None = None
    section: str | None = None
    excerpt: str = Field(min_length=1)


class Topic(BaseModel):
    id: str = Field(min_length=1)
    title: str = Field(min_length=1)
    summary: str = Field(min_length=1)
    context: str = Field(min_length=1)


class Concept(BaseModel):
    id: str = Field(min_length=1)
    topic_id: str = Field(min_length=1)
    title: str = Field(min_length=1)
    summary: str = Field(min_length=1)
    context: str = Field(min_length=1)
    prerequisite_ids: list[str] = Field(default_factory=list)
    evidence: list[Evidence] = Field(default_factory=list)


class DocumentNode(BaseModel):
    id: str = Field(min_length=1)
    title: str = Field(min_length=1)
    source_type: str = Field(min_length=1)
    domain: str = Field(min_length=1)
    overview: str = Field(min_length=1)


class StructuredGraph(BaseModel):
    document: DocumentNode
    topics: list[Topic]
    concepts: list[Concept]


In [31]:
def slugify(value: str) -> str:
    value = value.lower().strip()
    value = re.sub(r"[^a-z0-9]+", "-", value)
    return value.strip("-") or "document"


def read_document(path: str, max_pages: int = 10, max_chars: int = 24000) -> dict[str, str]:
    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(file_path)

    suffix = file_path.suffix.lower()
    if suffix == ".pdf":
        reader = PdfReader(str(file_path))
        pages = []
        for page in reader.pages[:max_pages]:
            pages.append(page.extract_text() or "")
        text = "\n\n".join(pages)
        source_type = "pdf"
    elif suffix in {".txt", ".md"}:
        text = file_path.read_text(encoding="utf-8")
        source_type = suffix.lstrip(".")
    else:
        raise ValueError("Supported file types in this notebook: .pdf, .txt, .md")

    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    if not text:
        raise ValueError(f"No text extracted from {file_path.name}")

    return {
        "document_id": slugify(file_path.stem),
        "title": file_path.stem.replace("_", " ").replace("-", " ").strip() or file_path.stem,
        "source_type": source_type,
        "text": text[:max_chars],
    }


In [32]:
GRAPH_PROMPT = """
You convert a bounded document into a minimal grounded graph.

Return valid JSON matching the StructuredGraph schema exactly.
Do not include markdown fences or commentary.

Rules:
- Create a small, useful graph from the provided document only.
- The document node must include:
  - domain: the subject area or discipline this material belongs to.
  - overview: a short explanation of what this material covers and why it matters.
- Topics are medium-granularity clusters from the document.
- Every topic must include:
  - summary: what the topic is.
  - context: how that topic functions in this specific document.
- Concepts are atomic ideas attached to one topic via topic_id.
- Every concept must include:
  - summary: what the concept means.
  - context: how that concept is used or framed in this specific document.
- prerequisite_ids must reference other concept ids when a prerequisite is clearly useful.
- If no clear prerequisite exists, use an empty list.
- Every concept should include 1-2 short evidence excerpts grounded in the source.
- Evidence may include page and section when available.
- Keep ids short, slug-like, and stable.
- Keep the graph minimal and practical: prefer 2-6 topics and 1-5 concepts per topic.
- Do not invent concepts that are not supported by the document.
"""

extractor_agent = Agent(
    model=model,
    name="GraphExtractor",
    instruction=GRAPH_PROMPT,
    output_schema=StructuredGraph,
    generate_content_config=types.GenerateContentConfig(temperature=0),
)

runner = Runner(
    agent=extractor_agent,
    app_name=f"{APP_NAME}_GRAPH",
    session_service=InMemorySessionService(),
)


In [33]:
async def build_graph(path: str, max_pages: int = 10) -> StructuredGraph:
    payload = read_document(path, max_pages=max_pages)
    session_id = str(uuid.uuid4())
    await runner.session_service.create_session(
        app_name=runner.app_name,
        user_id=USER_ID,
        session_id=session_id,
    )

    document_metadata = {
        "id": payload["document_id"],
        "title": payload["title"],
        "source_type": payload["source_type"],
    }
    prompt = (
        "Document metadata:\n"
        + json.dumps(document_metadata, ensure_ascii=False, indent=2)
        + "\n\nDocument text:\n"
        + payload["text"]
    )
    message = types.Content(role="user", parts=[types.Part(text=prompt)])

    final_text = None
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session_id,
        new_message=message,
    ):
        if event.is_final_response() and event.content and event.content.parts:
            texts = [
                part.text
                for part in event.content.parts
                if getattr(part, "text", None) and not getattr(part, "thought", False)
            ]
            final_text = "\n".join(texts).strip()

    if not final_text:
        raise RuntimeError("No final graph output received from the model.")

    return StructuredGraph.model_validate_json(final_text)


In [34]:
# Example:
graph = await build_graph("../data/sample_docs/Agent Quality.pdf", max_pages=10)
graph.model_dump()


Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 23 0 (offset 0)
Ignoring wrong pointing object 25 0 (offset 0)
Ignoring wrong pointing object 27 0 (offset 0)
Ignoring wrong pointing object 36 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)


{'document': {'id': 'agent-quality',
  'title': 'Agent Quality',
  'source_type': 'pdf',
  'domain': 'AI Software Engineering',
  'overview': 'A guide on transitioning from traditional deterministic quality assurance to a new paradigm for autonomous AI agents, emphasizing observability and continuous evaluation loops.'},
 'topics': [{'id': 'paradigm-shift',
   'title': 'The Paradigm Shift',
   'summary': 'The transition from predictable, instruction-based tools to autonomous, non-deterministic AI agents.',
   'context': 'Establishes the core problem: why traditional QA fails for AI agents and the need for a new approach to quality.'},
  {'id': 'observability',
   'title': 'Observability',
   'summary': "The technical foundation required to capture and analyze an agent's internal decision-making process.",
   'context': "Provides the implementation blueprint for making an agent 'evaluatable' through telemetry."},
  {'id': 'evaluation-strategies',
   'title': 'Evaluation Strategies',
   